# Who actually holds your money?

Most neobank comparisons rank apps by features. This one asks the question that decides whether your money is safe if the company dies: **is it a bank, and who is holding the deposits?**

The data is [neobankbeat](https://www.neobankbeat.com) — 368 verified-active neobanks, with the regulatory layer recorded per product. Full field dictionary and methodology: [neobankbeat.com/data/](https://www.neobankbeat.com/data/).

Three things fall out of it:

1. Roughly a third hold their own banking licence. The rest are split across e-money institutions, sponsor-bank models and software that never touches your money at all.
2. The web3-native cohort is a distinct wave arriving around 2023, not a continuation of the 2018 neobank boom — and in custody terms it is a different asset class.
3. Very few disclose user numbers, and the ones that do are extraordinarily concentrated.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Works both on Kaggle (dataset attached under /kaggle/input) and locally.
CANDIDATES = [
    Path("/kaggle/input/neobanks/entities.csv"),
    Path("entities.csv"),
    Path("../.staging/entities.csv"),
]
src = next((p for p in CANDIDATES if p.exists()), None)
if src is None:
    raise FileNotFoundError(
        "entities.csv not found. On Kaggle, add the neobankbeat/neobanks dataset "
        "via 'Add Input'. Locally, download it from "
        "https://www.kaggle.com/datasets/neobankbeat/neobanks"
    )
df = pd.read_csv(src)

plt.rcParams.update({"figure.figsize": (9, 5), "axes.spines.top": False, "axes.spines.right": False})
print(f"{len(df)} neobanks · {df.shape[1]} columns")
df.head(3)

## 1. How many are actually banks?

`regulation_type` is the field to reach for. A partner-bank (BaaS) model means the app is a software layer over someone else's licence — which is not inherently bad, but it is the difference between one regulator and two counterparties.

In [ ]:
reg = df["regulation_type"].fillna("Undisclosed").value_counts()

ax = reg.head(12).sort_values().plot.barh(color="#2563eb")
ax.set(title="How neobanks are authorised", xlabel="neobanks", ylabel="")
for i, v in enumerate(reg.head(12).sort_values()):
    ax.text(v + 1, i, str(v), va="center", fontsize=9)
plt.tight_layout()
plt.show()

reg.head(12).to_frame("count").assign(share=lambda t: (t["count"] / len(df)).map("{:.1%}".format))

## 2. Custody, cross-referenced with the licence

`custody` says who holds the money: a partner bank, the provider itself, or you (self-custody, for the on-chain apps). Crossing it with `category` shows how different the three models really are — self-custody means no deposit protection by construction, because there is no deposit.

In [ ]:
cross = pd.crosstab(df["category"], df["custody"].fillna("Undisclosed"))
cross = cross.loc[:, cross.sum().sort_values(ascending=False).index]

ax = cross.plot.bar(stacked=True, colormap="tab20", width=0.6)
ax.set(title="Custody model by category", xlabel="", ylabel="neobanks")
ax.legend(title="custody", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

cross

## 3. Founding cohorts: the crypto wave is visible

Plotting `founded` by `category` dates the shift. Traditional and hybrid neobanks share a median founding year of 2018; the web3-native median is 2023, which makes it a separate wave rather than a continuation.

In [ ]:
founded = df.dropna(subset=["founded"]).copy()
founded["founded"] = founded["founded"].astype(int)
cohorts = (
    founded[founded["founded"] >= 2005]
    .groupby(["founded", "category"])
    .size()
    .unstack(fill_value=0)
)

ax = cohorts.plot.area(alpha=0.85, colormap="viridis")
ax.set(title="Neobanks founded per year, by model", xlabel="year founded", ylabel="launches")
ax.legend(title="", loc="upper left")
plt.tight_layout()
plt.show()

print("Median founding year by category")
founded.groupby("category")["founded"].agg(["median", "count"]).sort_values("median")

## 4. Where you can actually open an account

`countries` is availability; `hq` is where the company sits. The two diverge sharply — a handful of hubs incorporate the companies, but availability is far wider and very uneven.

In [ ]:
def explode_semicolons(series):
    return (
        series.dropna()
        .str.split(";")
        .explode()
        .str.strip()
        .loc[lambda s: s.ne("")]
    )


avail = explode_semicolons(df["countries"]).value_counts()

ax = avail.head(15).sort_values().plot.barh(color="#0f766e")
ax.set(title="Countries with the most neobanks available", xlabel="neobanks available", ylabel="")
plt.tight_layout()
plt.show()

print(f"{avail.size} countries served in total")
print(f"Median neobanks per country: {avail.median():.0f}")
avail.head(15).to_frame("neobanks available")

## 5. What they actually let you do

`services` is a semicolon-separated capability list, verified per provider docs. Note the methodology caveat: tags are omitted when unverified, so absence of a tag is not proof the capability is missing. Read these as a floor, not a census.

In [ ]:
services = explode_semicolons(df["services"]).value_counts()

ax = services.sort_values().plot.barh(color="#7c3aed")
ax.set(title="Verified money-movement capabilities", xlabel="neobanks", ylabel="")
plt.tight_layout()
plt.show()

services.to_frame("neobanks")

## 6. Scale is extremely concentrated

`reported_users_millions` is self-disclosed and not audited, so treat it as order-of-magnitude — and check `reported_users_metric` before comparing two rows, because a wallet download is not a funded account.

In [ ]:
users = df.dropna(subset=["reported_users_millions"]).sort_values(
    "reported_users_millions", ascending=False
)
top = users.head(15)

ax = top.set_index("name")["reported_users_millions"].sort_values().plot.barh(color="#be123c")
ax.set(title="Largest by reported users (millions, self-disclosed)", xlabel="millions", ylabel="")
plt.tight_layout()
plt.show()

reported = users["reported_users_millions"]
print(f"{len(reported)} of {len(df)} disclose a user figure")
print(f"Top 10 account for {reported.nlargest(10).sum() / reported.sum():.0%} of all reported users")
top[["name", "reported_users_millions", "reported_users_metric", "reported_users_as_of", "regulation_type"]]

## 7. Build your own shortlist

The practical use of the dataset: filter on the things that matter to you. Here, licensed banks available in a given country, ranked by disclosed scale.

One trap worth pointing out, since it is easy to get wrong: do **not** filter `regulation_type` with a substring like `"licen"`. That also catches `Licence pending (partner model today)` and `VASP / MSB / crypto licences`, neither of which is a licensed bank. Match the category exactly.

In [ ]:
COUNTRY = "United Kingdom"  # try "Germany", "United States", "Brazil", "India"

available_here = df["countries"].fillna("").str.contains(COUNTRY, regex=False)
licensed = df["regulation_type"].eq("Licensed bank")

shortlist = df[available_here & licensed].sort_values(
    "reported_users_millions", ascending=False, na_position="last"
)
print(f"{len(shortlist)} licensed neobanks available in {COUNTRY}")
shortlist[["name", "regulation_type", "custody", "licence", "reported_users_millions", "website"]].head(20)

## Caveats and citation

- `reported_users_millions`, `funding` and `volume` are self-disclosed, not audited, and companies disclose selectively.
- `cashback`, `yield` and `fx_markup` are *up to* figures that vary by region and move constantly. Each carries an `as_of` date and a source — confirm with the issuer before relying on them.
- Coverage is live consumer-facing products: defunct neobanks and pure BaaS/infrastructure providers are excluded, so this is not a survival-analysis dataset.
- Unverified fields are empty rather than guessed.

Data: [neobankbeat](https://www.neobankbeat.com) (MIT). Nested source and as-of detail lives in `entities.jsonl`; this notebook uses the flattened `entities.csv`.

> neobankbeat (2026). *Open directory of neobanks worldwide.* https://www.neobankbeat.com/